# End-of-Text Attender

Attends primarily to the beginning-of-sequence / end-of-text token

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import torch as t
from IPython.display import Markdown, display
from shared import (
    load_model, run_and_cache, get_attention_pattern,
    show_head_pattern, show_attention_tables,
    compute_all_type_metrics, HEAD_TYPES, TYPE_ENTROPY_KEYS,
    ACTIVITY_LABELS, get_head_types, TEXT,
    show_type_tokens, show_type_filtered_tables,
    get_type_positions, TYPE_ID_TO_POSITION_KEY,
    populate_measurable_type_heads,
)

/home/burny/projects/ai/mechinterp/attention-head-zoo-2-layer-attention-only-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = load_model()
str_tokens, logits, cache = run_and_cache(model)
populate_measurable_type_heads(cache, str_tokens)

## Matched Tokens

In [3]:
show_type_tokens(str_tokens, "end_of_text")

**Matched tokens (1):** `<|endoftext|>` (0)

## End-of-Text Attender — All 24 Heads

In [4]:
tm = compute_all_type_metrics(cache, str_tokens)
ent_key = TYPE_ENTROPY_KEYS.get("end_of_text")
is_measurable = ("end_of_text", 0, 0) in tm
if is_measurable:
    results = sorted(
        [((l, h), tm[("end_of_text", l, h)]) for l in range(2) for h in range(12)],
        key=lambda x: x[1], reverse=True,
    )
    has_ent = ent_key and (ent_key, 0, 0) in tm
    if has_ent:
        lines = ["| Head | End-of-Text Attender % | Entropy % |", "|------|-------|-------|"]  
        for (l, h), pct in results:
            ent = tm[(ent_key, l, h)]
            lines.append(f"| L{l}H{h} | {pct:.2f}% | {ent:.2f}% |")
    else:
        lines = ["| Head | End-of-Text Attender % |", "|------|-------|"]  
        for (l, h), pct in results:
            lines.append(f"| L{l}H{h} | {pct:.2f}% |")
    display(Markdown("\n".join(lines)))
else:
    print("No programmatic metric available for this type.")

| Head | End-of-Text Attender % | Entropy % |
|------|-------|-------|
| L1H4 | 89.74% | 99.88% |
| L1H10 | 82.98% | 99.50% |
| L0H3 | 82.33% | 99.78% |
| L1H8 | 37.91% | 90.60% |
| L1H3 | 37.12% | 95.81% |
| L0H6 | 32.60% | 93.67% |
| L1H1 | 29.33% | 93.40% |
| L1H2 | 28.60% | 90.78% |
| L0H2 | 26.97% | 88.76% |
| L1H11 | 26.46% | 83.00% |
| L0H1 | 24.58% | 87.83% |
| L1H0 | 24.31% | 88.81% |
| L0H5 | 21.36% | 87.04% |
| L0H11 | 20.44% | 85.04% |
| L0H0 | 20.11% | 86.41% |
| L1H6 | 17.48% | 77.44% |
| L1H9 | 16.82% | 85.45% |
| L0H8 | 16.76% | 87.13% |
| L0H10 | 15.04% | 83.59% |
| L0H4 | 12.33% | 83.50% |
| L0H9 | 11.06% | 79.97% |
| L1H5 | 8.17% | 66.35% |
| L1H7 | 3.65% | 30.46% |
| L0H7 | 3.23% | 17.10% |

## Top Heads

In [5]:
pos_key = TYPE_ID_TO_POSITION_KEY.get("end_of_text")
_type_positions = get_type_positions(str_tokens).get(pos_key, []) if pos_key else []
sorted_heads = sorted(
    [((l, h), tm[("end_of_text", l, h)]) for l in range(2) for h in range(12) if ("end_of_text", l, h) in tm],
    key=lambda x: x[1], reverse=True,
)[:3]
for (l, h), pct in sorted_heads:
    ent_str = ""
    if ent_key and (ent_key, l, h) in tm:
        ent_str = f" | ent {tm[(ent_key, l, h)]:.2f}%"
    level = 'full' if pct >= 90 else 'fullish' if pct >= 60 else 'half' if pct >= 40 else 'partial' if pct >= 10 else 'almost none' if pct >= 0.1 else '-'
    display(Markdown(f"---\n### L{l}H{h} — {pct:.2f}% ({level}){ent_str}"))
    show_head_pattern(str_tokens, cache, layer=l, head=h)
    attention = get_attention_pattern(cache, layer=l, head=h)
    if _type_positions:
        show_type_filtered_tables(str_tokens, attention, _type_positions, "End-of-Text Attender", top_k=15)
    show_attention_tables(str_tokens, attention, top_k=25)

---
### L1H4 — 89.74% (fullish) | ent 99.88%

### Highest attention to End-of-Text Attender tokens (dest ← End-of-Text Attender src)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `<\|endoftext\|>` | `<\|endoftext\|>` | 0 | 0 | 1.0000 |
| 2 | ` machine` | `<\|endoftext\|>` | 9 | 0 | 0.9898 |
| 3 | ` powerful` | `<\|endoftext\|>` | 4 | 0 | 0.9856 |
| 4 | ` level` | `<\|endoftext\|>` | 32 | 0 | 0.9854 |
| 5 | ` super` | `<\|endoftext\|>` | 7 | 0 | 0.9847 |
| 6 | ` intelligence` | `<\|endoftext\|>` | 10 | 0 | 0.9813 |
| 7 | ` up` | `<\|endoftext\|>` | 29 | 0 | 0.9809 |
| 8 | ` century` | `<\|endoftext\|>` | 20 | 0 | 0.9780 |
| 9 | ` is` | `<\|endoftext\|>` | 11 | 0 | 0.9774 |
| 10 | ` learning` | `<\|endoftext\|>` | 25 | 0 | 0.9773 |
| 11 | ` solid` | `<\|endoftext\|>` | 52 | 0 | 0.9761 |
| 12 | `.` | `<\|endoftext\|>` | 21 | 0 | 0.9760 |
| 13 | ` techniques` | `<\|endoftext\|>` | 26 | 0 | 0.9744 |
| 14 | ` scaled` | `<\|endoftext\|>` | 28 | 0 | 0.9737 |
| 15 | ` systems` | `<\|endoftext\|>` | 41 | 0 | 0.9707 |

### Highest attention from End-of-Text Attender tokens (End-of-Text Attender dest ← src)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `<\|endoftext\|>` | `<\|endoftext\|>` | 0 | 0 | 1.0000 |

...

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `<\|endoftext\|>` | `<\|endoftext\|>` | 0 | 0 | 1.0000 |

### Highest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | `<\|endoftext\|>` | `<\|endoftext\|>` | 0 | 0 | 1.0000 |
| 2 | `We` | `<\|endoftext\|>` | 1 | 0 | 0.9974 |
| 3 | ` think` | `<\|endoftext\|>` | 2 | 0 | 0.9894 |
| 4 | ` powerful` | `<\|endoftext\|>` | 4 | 0 | 0.9892 |
| 5 | ` that` | `<\|endoftext\|>` | 3 | 0 | 0.9860 |
| 6 | ` super` | `<\|endoftext\|>` | 7 | 0 | 0.9708 |
| 7 | ` machine` | `<\|endoftext\|>` | 9 | 0 | 0.9694 |
| 8 | `human` | `<\|endoftext\|>` | 8 | 0 | 0.9678 |
| 9 | ` significantly` | `<\|endoftext\|>` | 6 | 0 | 0.9658 |
| 10 | ` intelligence` | `<\|endoftext\|>` | 10 | 0 | 0.9646 |
| 11 | ` is` | `<\|endoftext\|>` | 11 | 0 | 0.9445 |
| 12 | `,` | `<\|endoftext\|>` | 5 | 0 | 0.9434 |
| 13 | ` century` | `<\|endoftext\|>` | 20 | 0 | 0.9389 |
| 14 | ` likely` | `<\|endoftext\|>` | 13 | 0 | 0.9300 |
| 15 | ` created` | `<\|endoftext\|>` | 18 | 0 | 0.9274 |
| 16 | ` not` | `<\|endoftext\|>` | 15 | 0 | 0.9134 |
| 17 | `.` | `<\|endoftext\|>` | 21 | 0 | 0.9115 |
| 18 | ` than` | `<\|endoftext\|>` | 14 | 0 | 0.9106 |
| 19 | ` more` | `<\|endoftext\|>` | 12 | 0 | 0.9089 |
| 20 | ` current` | `<\|endoftext\|>` | 23 | 0 | 0.9065 |
| 21 | ` this` | `<\|endoftext\|>` | 19 | 0 | 0.9043 |
| 22 | ` to` | `<\|endoftext\|>` | 16 | 0 | 0.9024 |
| 23 | ` be` | `<\|endoftext\|>` | 17 | 0 | 0.9022 |
| 24 | ` machine` | `<\|endoftext\|>` | 24 | 0 | 0.8755 |
| 25 | ` learning` | `<\|endoftext\|>` | 25 | 0 | 0.8745 |

### Lowest attention weights (destination <- source)

| Rank | Dest Token | Src Token | Dest Pos | Src Pos | Weight |
|------|-----------|-----------|----------|---------|--------|
| 1 | ` known` | ` known` | 55 | 55 | 0.0006 |
| 2 | ` known` | ` not` | 55 | 15 | 0.0008 |
| 3 | ` plans` | ` not` | 53 | 15 | 0.0010 |
| 4 | `.` | `We` | 61 | 1 | 0.0010 |
| 5 | ` century` | `We` | 20 | 1 | 0.0010 |
| 6 | ` known` | ` is` | 55 | 11 | 0.0010 |
| 7 | ` current` | ` not` | 23 | 15 | 0.0010 |
| 8 | `ulative` | ` not` | 47 | 15 | 0.0010 |
| 9 | ` how` | ` not` | 57 | 15 | 0.0011 |
| 10 | ` level` | ` not` | 32 | 15 | 0.0012 |
| 11 | ` this` | ` known` | 60 | 55 | 0.0012 |
| 12 | ` plans` | ` is` | 53 | 11 | 0.0012 |
| 13 | `.` | `We` | 21 | 1 | 0.0012 |
| 14 | ` for` | ` not` | 56 | 15 | 0.0012 |
| 15 | ` powerful` | `We` | 4 | 1 | 0.0012 |
| 16 | ` or` | ` not` | 45 | 15 | 0.0012 |
| 17 | `human` | `We` | 8 | 1 | 0.0013 |
| 18 | ` known` | `We` | 55 | 1 | 0.0013 |
| 19 | ` systems` | ` not` | 41 | 15 | 0.0013 |
| 20 | ` techniques` | ` not` | 26 | 15 | 0.0013 |
| 21 | ` known` | ` up` | 55 | 29 | 0.0013 |
| 22 | ` century` | ` is` | 20 | 11 | 0.0013 |
| 23 | ` learning` | ` significantly` | 25 | 6 | 0.0013 |
| 24 | ` deceptive` | ` not` | 44 | 15 | 0.0013 |
| 25 | ` learning` | ` not` | 25 | 15 | 0.0013 |